# GLO-NCA V3 — Google Drive → GCS transfer (run in Google Colab)

Streams the BraTS-MET 2025 training ZIP **directly from your Google Drive into your GCS bucket** — no Windows upload, no anonymous datacenter download. The 33 GB file is streamed in chunks (never fully held in RAM or requiring 33 GB of Colab disk).

**How to run:** open this notebook in Colab (colab.research.google.com → Upload), then Runtime → Run all. When prompted, authorize (1) your Google account for Drive and (2) Google Cloud. Do the browser authorization in the Colab popups. **Do not paste any credentials anywhere else.**

Fixed facts (do not change): file id `1gkz4kM89PUdqI4rbpi1KN5TYGfBGYe5k`, bucket `gs://glo-nca-v3-even-continuity-501915`, project `even-continuity-501915-f9`.

In [ ]:
# Cell 1 — config + tools
FILE_ID   = '1gkz4kM89PUdqI4rbpi1KN5TYGfBGYe5k'
PROJECT   = 'even-continuity-501915-f9'
BUCKET    = 'glo-nca-v3-even-continuity-501915'
DEST_BLOB = 'datasets/brats-met-2025/MICCAI-LH-BraTS2025-MET-Challenge-TrainingData_batch1.zip'

# google-api-python-client + cloud storage are preinstalled on Colab, but pin/ensure:
!pip -q install --upgrade google-api-python-client google-cloud-storage >/dev/null 2>&1
import time, io
print('tools ready')

In [ ]:
# Cell 2 — authenticate Google (Drive + Cloud). Authorize in the popups.
from google.colab import auth
auth.authenticate_user()   # single Colab OAuth covers Drive API + GCS

import google.auth
creds, detected_project = google.auth.default()
print('authenticated; default project =', detected_project)

from googleapiclient.discovery import build
drive = build('drive', 'v3', credentials=creds)

from google.cloud import storage
gcs = storage.Client(project=PROJECT, credentials=creds)
print('Drive API + GCS client ready; using project', PROJECT)

In [ ]:
# Cell 3 — verify the Drive source (name, size, mime); do NOT trust name alone
meta = drive.files().get(fileId=FILE_ID,
                         fields='id,name,size,mimeType,md5Checksum',
                         supportsAllDrives=True).execute()
src_size = int(meta.get('size', 0))
print('id       :', meta['id'])
print('name     :', meta['name'])
print('mimeType :', meta['mimeType'])
print('size     : %d bytes (%.2f GB)' % (src_size, src_size/1e9))
print('md5      :', meta.get('md5Checksum', 'n/a'))
assert src_size > 30_000_000_000, 'unexpected size — not the full ~33 GB ZIP'
assert meta['name'].endswith('.zip'), 'source is not a .zip'
print('SOURCE OK')

In [ ]:
# Cell 4 — verify GCS bucket + destination; remove ONLY a prior partial of THIS object
bucket = gcs.bucket(BUCKET)
assert bucket.exists(), 'bucket %s not found' % BUCKET
print('bucket OK:', BUCKET, '| location:', bucket.location)
existing = bucket.get_blob(DEST_BLOB)
if existing is not None:
    print('existing object at destination: %d bytes' % (existing.size or 0))
    if (existing.size or 0) != src_size:
        print('  -> size != source; deleting incomplete/previous object before clean transfer')
        existing.delete()
        print('  deleted.')
    else:
        print('  -> already matches source size; transfer may be skipped (see Cell 5).')
else:
    print('destination clean (no existing object)')

In [ ]:
# Cell 5 — STREAM Drive -> GCS in chunks (no full RAM load, no 33 GB local file)
from googleapiclient.http import MediaIoBaseDownload

blob = bucket.blob(DEST_BLOB)
blob.content_type = 'application/zip'

already = bucket.get_blob(DEST_BLOB)
if already is not None and (already.size or 0) == src_size:
    print('destination already complete (%d bytes) — skipping transfer' % src_size)
else:
    CHUNK = 64 * 1024 * 1024  # 64 MB Drive read chunks
    req = drive.files().get_media(fileId=FILE_ID, supportsAllDrives=True)
    t0 = time.time()
    # Open a resumable GCS upload stream and pump Drive chunks straight into it.
    with blob.open('wb', content_type='application/zip', chunk_size=CHUNK) as gcs_out:
        buf = io.BytesIO()
        downloader = MediaIoBaseDownload(buf, req, chunksize=CHUNK)
        done = False
        transferred = 0
        while not done:
            status, done = downloader.next_chunk(num_retries=5)
            data = buf.getvalue()
            if data:
                gcs_out.write(data)
                transferred += len(data)
                buf.seek(0); buf.truncate(0)
            el = time.time() - t0
            spd = transferred / el / 1e6 if el > 0 else 0
            pct = 100.0 * transferred / src_size if src_size else 0
            eta = (src_size - transferred) / (transferred/el) / 60 if transferred and el else 0
            print('  %.1f%%  %.2f/%.2f GB  %.0f MB/s  elapsed %.1f min  ETA %.1f min'
                  % (pct, transferred/1e9, src_size/1e9, spd, el/60, eta), flush=True)
    print('TRANSFER DONE in %.1f min' % ((time.time()-t0)/60))

In [ ]:
# Cell 6 — verify the GCS object (size match + ZIP magic bytes via a 4-byte ranged read)
blob = bucket.get_blob(DEST_BLOB)
assert blob is not None, 'destination object missing after transfer'
dst_size = blob.size
print('destination:', 'gs://%s/%s' % (BUCKET, DEST_BLOB))
print('dst size   : %d bytes (%.2f GB)' % (dst_size, dst_size/1e9))
print('src size   : %d bytes' % src_size)
print('generation :', blob.generation, '| content_type:', blob.content_type)
size_ok = (dst_size == src_size)
magic = blob.download_as_bytes(start=0, end=3)  # first 4 bytes only
zip_ok = magic[:2] == b'PK'
print('SIZE MATCH :', size_ok)
print('ZIP MAGIC  :', magic, '->', zip_ok)
assert size_ok and zip_ok, 'VERIFICATION FAILED (size or magic mismatch)'
print('GCS VERIFICATION: PASS')

In [ ]:
# Cell 7 — final report
print('='*56)
print('GLO-NCA V3 — Drive -> GCS transfer report')
print('='*56)
print('SOURCE            : Google Drive file', FILE_ID)
print('SOURCE NAME       :', meta['name'])
print('DESTINATION       : gs://%s/%s' % (BUCKET, DEST_BLOB))
print('SIZE (src=dst)    : %d bytes (%.2f GB)' % (dst_size, dst_size/1e9))
print('CONTENT TYPE      :', blob.content_type)
print('GENERATION        :', blob.generation)
print('AUTH              : Colab authenticate_user (no tokens exposed)')
print('PROJECT           :', PROJECT)
print('BUCKET LOCATION   :', bucket.location)
print('GCS VERIFICATION  :', 'PASS' if (size_ok and zip_ok) else 'FAIL')
print('DATA TRANSFER GATE:', 'PASS' if (size_ok and zip_ok) else 'BLOCKED')
print('GPU               : NOT STARTED')
print('TRAINING          : NOT STARTED')
print('='*56)
print('Next: on the GCP training VM, copy this ZIP from GCS, extract, and run')
print('scripts/validate_dataset.py + scripts/check_split.py (separate task).')